# **Customer Journey Analysis**
**Context:** How customers move from advertising sources to completing a purchase.

**Objective:** Build a multi-stage Sankey diagram to answer the following questions:

From different marketing sources (utm_source), which pages do users land on first (landing_page)?
From those pages, which products (product_name) do they purchase, or do they leave the website (Drop-off)?

## **1. Import Library**

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from google.colab import drive


## **2. Load the Dataset**

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
orders= pd.read_csv('/content/drive/MyDrive/Xom Analytics class/Buoi 9 Sankey Flow/Buoi 9 HW/orders (2).csv')
products= pd.read_csv('/content/drive/MyDrive/Xom Analytics class/Buoi 9 Sankey Flow/Buoi 9 HW/products (2).csv')
website_pageviews= pd.read_csv('/content/drive/MyDrive/Xom Analytics class/Buoi 9 Sankey Flow/Buoi 9 HW/website_pageviews (2).csv')
website_sessions= pd.read_csv('/content/drive/MyDrive/Xom Analytics class/Buoi 9 Sankey Flow/Buoi 9 HW/website_sessions (2).csv')

## **3. Customer Journey Analysis**

### 3.1 Identify Landing Page

In [4]:
df_pageviews = website_pageviews.sort_values(['website_session_id', 'created_at'])
df_pageviews.head(
)

,website_pageview_id,created_at,website_session_id,pageview_url
0,1,2012-03-19 08:04:16,1,/home
1,2,2012-03-19 08:16:49,2,/home
2,3,2012-03-19 08:26:55,3,/home
3,4,2012-03-19 08:37:33,4,/home
4,5,2012-03-19 09:00:55,5,/home


In [5]:
landing_page = df_pageviews.groupby('website_session_id').first().reset_index()
landing_page = landing_page[['website_session_id', 'pageview_url']]
landing_page.rename(columns={'pageview_url': 'landing_page'}, inplace=True)
landing_page.head(20)


,website_session_id,landing_page
0,1,/home
1,2,/home
2,3,/home
3,4,/home
4,5,/home
5,6,/home
6,7,/home
7,8,/home
8,9,/home
9,10,/home


### 3.2 Multi-table join

In [6]:
#Merge website_sessions (utm_source) vs landing_page
df_merged = website_sessions[['website_session_id', 'utm_source']].merge(
    landing_page[['website_session_id', 'landing_page']],
    on='website_session_id',
    how='left'
)
df_merged.head(20)

,website_session_id,utm_source,landing_page
0,1,gsearch,/home
1,2,gsearch,/home
2,3,gsearch,/home
3,4,gsearch,/home
4,5,gsearch,/home
5,6,gsearch,/home
6,7,gsearch,/home
7,8,gsearch,/home
8,9,gsearch,/home
9,10,gsearch,/home


In [7]:
# Merge with orders
df_merged = df_merged.merge(
    orders[['website_session_id', 'primary_product_id']],
    on='website_session_id',
    how='left'
)
df_merged.head(20)

,website_session_id,utm_source,landing_page,primary_product_id
0,1,gsearch,/home,NaN
1,2,gsearch,/home,NaN
2,3,gsearch,/home,NaN
3,4,gsearch,/home,NaN
4,5,gsearch,/home,NaN
5,6,gsearch,/home,NaN
6,7,gsearch,/home,NaN
7,8,gsearch,/home,NaN
8,9,gsearch,/home,NaN
9,10,gsearch,/home,NaN


In [8]:
# Merge vith products to get product_name
df_merged = df_merged.merge(
    products[['product_id', 'product_name']],
    left_on='primary_product_id',
    right_on='product_id',
    how='left'
)
df_merged

,website_session_id,utm_source,landing_page,primary_product_id,product_id,product_name
0,1,gsearch,/home,NaN,NaN,NaN
1,2,gsearch,/home,NaN,NaN,NaN
2,3,gsearch,/home,NaN,NaN,NaN
3,4,gsearch,/home,NaN,NaN,NaN
4,5,gsearch,/home,NaN,NaN,NaN
...,...,...,...,...,...,...
472866,472867,gsearch,/home,NaN,NaN,NaN
472867,472868,bsearch,/lander-3,NaN,NaN,NaN
472868,472869,gsearch,/lander-3,NaN,NaN,NaN
472869,472870,gsearch,/lander-5,NaN,NaN,NaN


### 3.3 Handle Drop-off Data

In [9]:
df_merged['product_name'] = df_merged['product_name'].fillna('None (Drop-off)')

df_merged.head()

,website_session_id,utm_source,landing_page,primary_product_id,product_id,product_name
0,1,gsearch,/home,NaN,NaN,None (Drop-off)
1,2,gsearch,/home,NaN,NaN,None (Drop-off)
2,3,gsearch,/home,NaN,NaN,None (Drop-off)
3,4,gsearch,/home,NaN,NaN,None (Drop-off)
4,5,gsearch,/home,NaN,NaN,None (Drop-off)


### 3.4. Sankey Flow

Sankey stage 1

In [10]:
df_sankey_stage1 = df_merged.groupby(['utm_source', 'landing_page']).size().reset_index(name='value')
df_sankey_stage1

,utm_source,landing_page,value
0,bsearch,/home,7914
1,bsearch,/lander-1,9459
2,bsearch,/lander-2,24076
3,bsearch,/lander-3,6178
4,bsearch,/lander-4,1903
5,bsearch,/lander-5,13293
6,gsearch,/home,46334
7,gsearch,/lander-1,38115
8,gsearch,/lander-2,100982
9,gsearch,/lander-3,68249


In [11]:
df_sankey_stage1.rename(columns={'utm_source': 'source', 'landing_page': 'target'}, inplace=True)
df_sankey_stage1.head()

,source,target,value
0,bsearch,/home,7914
1,bsearch,/lander-1,9459
2,bsearch,/lander-2,24076
3,bsearch,/lander-3,6178
4,bsearch,/lander-4,1903


Sankey stage 2

In [14]:
df_sankey_stage2 = df_merged.groupby([ 'landing_page', 'product_name']).size().reset_index(name='value')
df_sankey_stage2.rename(columns={'landing_page': 'source', 'product_name': 'target'}, inplace=True)
df_sankey_stage2

,source,target,value
0,/home,None (Drop-off),127865
1,/home,The Birthday Sugar Panda,1003
2,/home,The Forever Love Bear,1458
3,/home,The Hudson River Mini bear,182
4,/home,The Original Mr. Fuzzy,7068
5,/lander-1,None (Drop-off),45417
6,/lander-1,The Forever Love Bear,92
7,/lander-1,The Original Mr. Fuzzy,2065
8,/lander-2,None (Drop-off),121042
9,/lander-2,The Birthday Sugar Panda,791


In [15]:
df_sankey = pd.concat([df_sankey_stage1, df_sankey_stage2], ignore_index=True)


### 3.5 Visualization

In [16]:
# Create a list of all unique nodes (sources and targets) in the entire flow
all_nodes = pd.concat([
    df_sankey['source'],
    df_sankey['target']
]).unique()

# Create a mapping from node names to unique integer indices, which Plotly Sankey requires
node_to_index = {node: i for i, node in enumerate(all_nodes)}

# Map the 'source' and 'target' columns in the combined dataframe to their integer indices
source_indices = df_sankey['source'].map(node_to_index).tolist()
target_indices = df_sankey['target'].map(node_to_index).tolist()
value = df_sankey['value'].tolist()

# Create the Sankey diagram using plotly.graph_objects
fig = go.Figure(data=[
    go.Sankey(
        node=dict(
            pad=15, # Padding between nodes
            thickness=20, # Thickness of the nodes
            line=dict(color="black", width=0.5), # Node border styling
            label=all_nodes, # Labels for the nodes
            color="blue" # Color of the nodes (can be an array for different colors)
        ),
        link=dict(
            source=source_indices, # Indices of the source nodes
            target=target_indices, # Indices of the target nodes
            value=value, # Values of the flows
            # color="lightgray" # Color of the links (can be an array for different colors)
        )
    )
])

# Update the layout of the figure with a title and font size
fig.update_layout(
    title_text="Customer Journey Analysis: UTM Source -> Landing Page -> Product",
    font_size=10
)

# Display the figure
fig.show()

## **4**. Insights

In [17]:
# Q1: Which traffic source has the highest order conversion rate?
conv_by_source = df_merged.groupby('utm_source').agg(
    total_sessions=('website_session_id', 'count'),
    orders=('product_name', lambda x: (x != 'None (Drop-off)').sum())
).reset_index()
conv_by_source['conversion_rate_%'] = (conv_by_source['orders'] / conv_by_source['total_sessions'] * 100).round(2)
conv_by_source = conv_by_source.sort_values('conversion_rate_%', ascending=False)
conv_by_source

,utm_source,total_sessions,orders,conversion_rate_%
0,bsearch,62823,4519,7.19
1,gsearch,316035,21333,6.75
2,socialbook,10685,343,3.21


 **bsearch has the highest conversion rate at 7.19%**, slightly outperforming gsearch (6.75%). socialbook lags far behind at just 3.21%, despite being a distinct traffic channel. While gsearch drives by far the most volume (315K sessions), bsearch users are more purchase-intent-driven per session

In [18]:
# Q2: Landing pages with high traffic but high drop-off rate?
lp_analysis = df_merged.groupby('landing_page').agg(
    total_sessions=('website_session_id', 'count'),
    dropoffs=('product_name', lambda x: (x == 'None (Drop-off)').sum()),
    orders=('product_name', lambda x: (x != 'None (Drop-off)').sum())
).reset_index()
lp_analysis['dropoff_rate_%'] = (lp_analysis['dropoffs'] / lp_analysis['total_sessions'] * 100).round(2)
lp_analysis['conversion_rate_%'] = (lp_analysis['orders'] / lp_analysis['total_sessions'] * 100).round(2)
lp_analysis = lp_analysis.sort_values('total_sessions', ascending=False)
lp_analysis

,landing_page,total_sessions,dropoffs,orders,dropoff_rate_%,conversion_rate_%
0,/home,137576,127865,9711,92.94,7.06
2,/lander-2,131170,121042,10128,92.28,7.72
3,/lander-3,79000,76321,2679,96.61,3.39
5,/lander-5,68166,61236,6930,89.83,10.17
1,/lander-1,47574,45417,2157,95.47,4.53
4,/lander-4,9385,8677,708,92.46,7.54


**lander-3 is the biggest problem:** it receives 79,000 sessions but converts only 3.39% — meaning nearly 97 out of every 100 visitors leave without buying anything. Given its high traffic volume, fixing this page would have an outsized business impact.

**home is also notable:** it's the most visited page (137K sessions) yet converts at only 7.06%, losing ~128K potential buyers.

**lander-5 is the standout performer with the lowest drop-off** (89.83%) and highest conversion (10.17%) — its design or audience targeting should be studied and replicated.

In [21]:
#Q3 The difference in shopping behavior between customers coming from gsearch and bsearch
# Overall conversion
search_df = df_merged[df_merged['utm_source'].isin(['gsearch', 'bsearch'])].copy()
search_conv = search_df.groupby('utm_source').agg(
    total_sessions=('website_session_id', 'count'),
    orders=('product_name', lambda x: (x != 'None (Drop-off)').sum())
).reset_index()
search_conv['conversion_rate_%'] = (search_conv['orders'] / search_conv['total_sessions'] * 100).round(2)
search_conv

,utm_source,total_sessions,orders,conversion_rate_%
0,bsearch,62823,4519,7.19
1,gsearch,316035,21333,6.75


In [23]:
#Product preference
prod_pref = (search_df[search_df['product_name'] != 'None (Drop-off)']
             .groupby(['utm_source', 'product_name']).size().reset_index(name='orders'))
prod_pref['pct_of_orders_%'] = (prod_pref['orders'] / prod_pref.groupby('utm_source')['orders'].transform('sum') * 100).round(2)
prod_pref = prod_pref.sort_values(['utm_source', 'orders'], ascending=[True, False])
prod_pref

,utm_source,product_name,orders,pct_of_orders_%
3,bsearch,The Original Mr. Fuzzy,3360,74.35
1,bsearch,The Forever Love Bear,648,14.34
0,bsearch,The Birthday Sugar Panda,425,9.40
2,bsearch,The Hudson River Mini bear,86,1.90
7,gsearch,The Original Mr. Fuzzy,15853,74.31
5,gsearch,The Forever Love Bear,3152,14.78
4,gsearch,The Birthday Sugar Panda,1951,9.15
6,gsearch,The Hudson River Mini bear,377,1.77


In [25]:
lp_dist = search_df.groupby(['utm_source', 'landing_page']).size().reset_index(name='sessions')
lp_dist['pct_%'] = (lp_dist['sessions'] / lp_dist.groupby('utm_source')['sessions'].transform('sum') * 100).round(2)
lp_dist = lp_dist.sort_values(['utm_source', 'sessions'], ascending=[True, False])
lp_dist

,utm_source,landing_page,sessions,pct_%
2,bsearch,/lander-2,24076,38.32
5,bsearch,/lander-5,13293,21.16
1,bsearch,/lander-1,9459,15.06
0,bsearch,/home,7914,12.60
3,bsearch,/lander-3,6178,9.83
4,bsearch,/lander-4,1903,3.03
8,gsearch,/lander-2,100982,31.95
9,gsearch,/lander-3,68249,21.60
11,gsearch,/lander-5,54873,17.36
6,gsearch,/home,46334,14.66


**Bsearch** is the **more efficient traffic source** because it has a slightly higher conversion rate and attracts users who appear more ready to purchase. In contrast, **gsearch** drives **much higher traffic volume**, making it valuable for reach and awareness, but its lower conversion rate suggests more room for landing page optimization. Both channels show similar product preferences, with **The Original Mr. Fuzzy clearly dominating completed orders**. Since gsearch traffic is spread across more landing pages, especially /lander-3 and /lander-5, these pages should be reviewed to reduce drop-off and improve conversion. Overall, marketing should use **gsearch for scale, bsearch for high-intent traffic**, and position **The Original Mr. Fuzzy** as the main product hook across both channels.